# Day 3: Externalized Prompt Injection & Local SLM Inference
**Config Source:** `.env` + `configs/app_config.yaml` (`engine/settings.py`) — Zero hardcoded URLs in notebook cells.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
ACTIVE_DOMAIN = SETTINGS.active_domain  # Override here or via BOOTCAMP_DOMAIN in .env

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Externalized YAML Prompts & Deterministic Decoding**
Hot-reloads `configs/prompts/day03_summarizer.yaml` and runs offline `Qwen2.5-Coder-1.5B` without hardcoded prompts or URLs.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
dbg.start_run("day03_inference")
with dbg.start_span(f"Day3_Inference_{ACTIVE_INCIDENT['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    prompt_cfg = prompts.load("day03_summarizer")
    hydrated_user_prompt = prompts.render(prompt_cfg["user_template"], incident_text=ACTIVE_INCIDENT["raw_log"])
    t_prompt = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("load_yaml_prompt", {"template": "day03_summarizer.yaml"}, hydrated_user_prompt, t_prompt, span_kind="CHAIN")

    res = llm.generate(
        prompt=hydrated_user_prompt,
        system_instruction=prompt_cfg["system_prompt"],
        temperature=float(prompt_cfg["temperature"]),
        max_tokens=int(prompt_cfg["max_tokens"]),
    )
    dbg.record_step("qwen_deterministic_gen", hydrated_user_prompt, res["output"], res["latency_ms"], span_kind="LLM")

viz.render_story(
    title=f"Day 3 Externalized Prompt & SLM Inference — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "YAML Prompt Hydrated", "badge": "PROMPT REGISTRY", "latency_ms": t_prompt, "summary": f"temp={prompt_cfg['temperature']}", "payload": hydrated_user_prompt},
        {"name": f"Offline Generation ({res['model']})", "badge": "LOCAL SLM", "latency_ms": res["latency_ms"], "summary": "Zero external URLs or cloud calls", "payload": res["output"]}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Gemini inference parity active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local Qwen inference verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert len(res["output"]) > 5
print("[PASS] Day 3 Inference Engineering Gate passed!")
